In [1]:
import os
import math
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

try:
    from tqdm import tqdm
except ModuleNotFoundError:
    tqdm = lambda x, **kwargs: x

In [2]:
!pwd

/Users/ronit/Desktop/QeFEM/main/maxcut_results/v2


In [3]:
K2000_HIST_BEST = 33337

In [6]:
DATA_MAXCUT_DIR = Path("./../../data-maxcut")
RESULTS_DIR = Path(".")
OUT_DIR = RESULTS_DIR / "full_res"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GSET_HIST_PATH = RESULTS_DIR / "gset_validation.txt"
GSET_VANILLA_PATH = RESULTS_DIR / "naive_vanilla_fem_G1-G54_full_results.csv"
GSET_QEFEM_V1_PATH = RESULTS_DIR / "qefem_v1_benchmark_table.csv"
K2000_RESULTS_PATH = RESULTS_DIR / "K2000_adam_anneal_vs_baselines_comparison.csv"

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32

In [7]:
def build_gset_history_df(hist_path=GSET_HIST_PATH, out_dir=OUT_DIR):
    rows = []

    with open(hist_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            g_idx, best_cut = line.split()[:2]
            g_idx = int(g_idx)

            if 1 <= g_idx <= 54:
                rows.append({
                    "instance": f"G{g_idx}",
                    "best_known_cut": float(best_cut),
                })

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "gset_history_best_G1-G54.csv", index=False)
    return df

def build_gset_vanilla_best_df(vanilla_path=GSET_VANILLA_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(vanilla_path)

    df = df_raw.loc[:, ["instance", "vanilla_fem_best_cut", "gap"]].copy()
    df = df.rename(columns={
        "instance": "graph_name",
        "vanilla_fem_best_cut": "cut_value",
    })

    df.to_csv(out_dir / "gset_vanilla_best_G1-G54.csv", index=False)
    return df

def build_gset_qefem_v1_best_df(qefem_v1_path=GSET_QEFEM_V1_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(qefem_v1_path)

    df = df_raw.copy()
    df["g_idx"] = df["instance"].str.replace("G", "", regex=False).astype(int)
    df = df[(df["g_idx"] >= 1) & (df["g_idx"] <= 54)]

    df = df.loc[:, ["instance", "v1_best", "v1_gap"]].copy()
    df = df.rename(columns={
        "instance": "graph_name",
        "v1_best": "cut_value",
        "v1_gap": "gap",
    })

    df.to_csv(out_dir / "gset_qefem_v1_best_G1-G54.csv", index=False)
    return df

def build_k2000_vanilla_best_df(k2000_path=K2000_RESULTS_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(k2000_path)

    df = df_raw[df_raw["method"].str.startswith("vanilla")].copy()
    df = df.loc[:, ["method", "cut", "gap"]]
    df = df.rename(columns={
        "method": "graph_name",
        "cut": "cut_value",
    })

    df.to_csv(out_dir / "k2000_vanilla_best.csv", index=False)
    return df

def build_k2000_qefem_v1_best_df(k2000_path=K2000_RESULTS_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(k2000_path)

    df = df_raw[df_raw["method"].str.startswith("QEFEM")].copy()
    df = df.loc[:, ["method", "cut", "gap"]]
    df = df.rename(columns={
        "method": "graph_name",
        "cut": "cut_value",
    })

    df.to_csv(out_dir / "k2000_qefem_v1_best.csv", index=False)
    return df

def build_all_reference_dfs():
    gset_history_df = build_gset_history_df()
    gset_vanilla_df = build_gset_vanilla_best_df()
    gset_qefem_v1_df = build_gset_qefem_v1_best_df()
    k2000_vanilla_df = build_k2000_vanilla_best_df()
    k2000_qefem_v1_df = build_k2000_qefem_v1_best_df()

    return {
        "gset_history": gset_history_df,
        "gset_vanilla": gset_vanilla_df,
        "gset_qefem_v1": gset_qefem_v1_df,
        "k2000_vanilla": k2000_vanilla_df,
        "k2000_qefem_v1": k2000_qefem_v1_df,
    }

In [9]:
def build_gset_history_df(hist_path=GSET_HIST_PATH, out_dir=OUT_DIR):
    rows = []

    with open(hist_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            g_idx, best_cut = line.split()[:2]
            g_idx = int(g_idx)

            if 1 <= g_idx <= 54:
                rows.append({
                    "instance": f"G{g_idx}",
                    "best_known_cut": float(best_cut),
                })

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "gset_history_best_G1-G54.csv", index=False)
    return df

def build_gset_vanilla_best_df(vanilla_path=GSET_VANILLA_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(vanilla_path)

    df = df_raw.loc[:, ["instance", "vanilla_fem_best_cut", "gap"]].copy()
    df = df.rename(columns={
        "instance": "graph_name",
        "vanilla_fem_best_cut": "cut_value",
    })

    df.to_csv(out_dir / "gset_vanilla_best_G1-G54.csv", index=False)
    return df

def build_gset_qefem_v1_best_df(qefem_v1_path=GSET_QEFEM_V1_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(qefem_v1_path)

    df = df_raw.copy()
    df["g_idx"] = df["instance"].str.replace("G", "", regex=False).astype(int)
    df = df[(df["g_idx"] >= 1) & (df["g_idx"] <= 54)]

    df = df.loc[:, ["instance", "v1_best", "v1_gap"]].copy()
    df = df.rename(columns={
        "instance": "graph_name",
        "v1_best": "cut_value",
        "v1_gap": "gap",
    })

    df.to_csv(out_dir / "gset_qefem_v1_best_G1-G54.csv", index=False)
    return df

def build_k2000_vanilla_best_df(k2000_path=K2000_RESULTS_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(k2000_path)

    df = df_raw[df_raw["method"].str.startswith("vanilla")].copy()
    df = df.loc[:, ["method", "cut", "gap"]]
    df = df.rename(columns={
        "method": "graph_name",
        "cut": "cut_value",
    })

    df.to_csv(out_dir / "k2000_vanilla_best.csv", index=False)
    return df

def build_k2000_qefem_v1_best_df(k2000_path=K2000_RESULTS_PATH, out_dir=OUT_DIR):
    df_raw = pd.read_csv(k2000_path)

    df = df_raw[df_raw["method"].str.startswith("QEFEM")].copy()
    df = df.loc[:, ["method", "cut", "gap"]]
    df = df.rename(columns={
        "method": "graph_name",
        "cut": "cut_value",
    })

    df.to_csv(out_dir / "k2000_qefem_v1_best.csv", index=False)
    return df

def build_all_reference_dfs():
    gset_history_df = build_gset_history_df()
    gset_vanilla_df = build_gset_vanilla_best_df()
    gset_qefem_v1_df = build_gset_qefem_v1_best_df()
    k2000_vanilla_df = build_k2000_vanilla_best_df()
    k2000_qefem_v1_df = build_k2000_qefem_v1_best_df()

    return {
        "gset_history": gset_history_df,
        "gset_vanilla": gset_vanilla_df,
        "gset_qefem_v1": gset_qefem_v1_df,
        "k2000_vanilla": k2000_vanilla_df,
        "k2000_qefem_v1": k2000_qefem_v1_df,
    }

reference_dfs = build_all_reference_dfs()

In [10]:
def load_gset_dense(g_idx, data_dir=DATA_MAXCUT_DIR, device=DEVICE, dtype=DTYPE):
    if not (1 <= int(g_idx) <= 54):
        raise ValueError("g_idx must be between 1 and 54.")

    path = data_dir / f"G{int(g_idx)}"

    with open(path, "r") as f:
        n, m = map(int, f.readline().split())

        W_cpu = torch.zeros((n, n), dtype=dtype, device="cpu")

        for _ in range(m):
            i, j, w = f.readline().split()
            i = int(i) - 1
            j = int(j) - 1
            w = float(w)

            W_cpu[i, j] = w
            W_cpu[j, i] = w

    return W_cpu.to(device=device)

def load_k2000_dense(data_dir=DATA_MAXCUT_DIR, device=DEVICE, dtype=DTYPE):
    path = data_dir / "k2000"

    with open(path, "r") as f:
        n, m = map(int, f.readline().split())

        W_cpu = torch.zeros((n, n), dtype=dtype, device="cpu")

        for _ in range(m):
            i, j, w = f.readline().split()
            i = int(i) - 1
            j = int(j) - 1
            w = float(w)

            W_cpu[i, j] = w
            W_cpu[j, i] = w

    return W_cpu.to(device=device)

In [11]:
EPS = 1e-8
GRAD_CLIP_DEFAULT = 1.0
BLOCH_LENGTH_INIT = 1e-2
FIELD_NOISE = 1e-3
THETA_INIT = math.pi / 2
THETA_NOISE = 1e-3

In [12]:
def print_gset_hparam_help():
    print(
        """
get_gset_local_hparams input sequence:

Required:
    g_idx

Common tunables:
    n_batch
    n_rep
    n_steps
    optimizer
    lr
    grad_type
    grad_clip
    T_max
    T_min
    beta_schedule
    gamma_max
    gamma_min
    gamma_schedule

Adam tunables:
    adam_betas
    adam_weight_decay

RMSprop tunables:
    rmsprop_alpha
    rmsprop_momentum
    rmsprop_weight_decay

Example:
    hp = get_gset_local_hparams(
        4,
        n_batch=10000,
        n_rep=1,
        n_steps=1000,
        optimizer="adam",
        lr=1e-2,
        grad_type="explicit",
        T_max=0.5,
        T_min=8e-5,
        beta_schedule="exponential_beta",
        gamma_max=1.0,
        gamma_min=1e-3,
        gamma_schedule="exponential",
    )
"""
    )


def get_gset_local_hparams(
    g_idx,
    *,
    n_batch=10000,
    n_rep=1,
    n_steps=1000,

    optimizer="adam",
    lr=1e-2,

    adam_betas=(0.9, 0.999),
    adam_weight_decay=0.0,

    rmsprop_alpha=0.99,
    rmsprop_momentum=0.0,
    rmsprop_weight_decay=0.0,

    grad_type="auto",
    grad_clip=GRAD_CLIP_DEFAULT,

    T_max=0.5,
    T_min=8e-5,
    beta_schedule="exponential_beta",

    gamma_max=1.0,
    gamma_min=1e-3,
    gamma_schedule="exponential",

    show_help=False,
):
    if show_help:
        print_gset_hparam_help()

    if not isinstance(g_idx, int):
        print("ERROR: g_idx must be an integer.")
        print_gset_hparam_help()
        raise TypeError("g_idx must be an integer.")

    if g_idx < 1 or g_idx > 54:
        print("ERROR: g_idx must be between 1 and 54.")
        print_gset_hparam_help()
        raise ValueError("g_idx must be between 1 and 54.")

    optimizer = optimizer.lower()
    grad_type = grad_type.lower()
    beta_schedule = beta_schedule.lower()
    gamma_schedule = gamma_schedule.lower()

    if optimizer not in ["adam", "rmsprop"]:
        print("ERROR: optimizer must be either 'adam' or 'rmsprop'.")
        print_gset_hparam_help()
        raise ValueError("Invalid optimizer.")

    if grad_type not in ["auto", "explicit"]:
        print("ERROR: grad_type must be either 'auto' or 'explicit'.")
        print_gset_hparam_help()
        raise ValueError("Invalid grad_type.")

    if beta_schedule not in ["linear_temperature", "linear_beta", "exponential_beta"]:
        print("ERROR: beta_schedule must be one of:")
        print("    'linear_temperature', 'linear_beta', 'exponential_beta'")
        print_gset_hparam_help()
        raise ValueError("Invalid beta_schedule.")

    if gamma_schedule not in ["linear", "exponential", "constant"]:
        print("ERROR: gamma_schedule must be one of:")
        print("    'linear', 'exponential', 'constant'")
        print_gset_hparam_help()
        raise ValueError("Invalid gamma_schedule.")

    if n_batch <= 0:
        print("ERROR: n_batch must be positive.")
        print_gset_hparam_help()
        raise ValueError("Invalid n_batch.")

    if n_rep <= 0:
        print("ERROR: n_rep must be positive.")
        print_gset_hparam_help()
        raise ValueError("Invalid n_rep.")

    if n_steps <= 0:
        print("ERROR: n_steps must be positive.")
        print_gset_hparam_help()
        raise ValueError("Invalid n_steps.")

    if lr <= 0:
        print("ERROR: lr must be positive.")
        print_gset_hparam_help()
        raise ValueError("Invalid lr.")

    if T_max <= 0 or T_min <= 0:
        print("ERROR: T_max and T_min must both be positive.")
        print_gset_hparam_help()
        raise ValueError("Invalid temperature range.")

    if gamma_max < 0 or gamma_min < 0:
        print("ERROR: gamma_max and gamma_min must be non-negative.")
        print_gset_hparam_help()
        raise ValueError("Invalid gamma range.")

    return {
        "graph_name": f"G{g_idx}",

        "n_batch": int(n_batch),
        "n_rep": int(n_rep),
        "n_steps": int(n_steps),

        "optimizer": optimizer,
        "lr": float(lr),

        "adam_betas": adam_betas,
        "adam_weight_decay": float(adam_weight_decay),

        "rmsprop_alpha": float(rmsprop_alpha),
        "rmsprop_momentum": float(rmsprop_momentum),
        "rmsprop_weight_decay": float(rmsprop_weight_decay),

        "grad_type": grad_type,
        "gradient_mode": grad_type,
        "grad_clip": grad_clip,

        "T_max": float(T_max),
        "T_min": float(T_min),
        "beta_schedule": beta_schedule,

        "gamma_max": float(gamma_max),
        "gamma_min": float(gamma_min),
        "gamma_schedule": gamma_schedule,
    }

In [13]:
def print_k2000_hparam_help():
    print(
        """
get_k2000_local_hparams input sequence:

Common tunables:
    n_batch
    n_rep
    n_steps
    optimizer
    lr
    grad_type
    grad_clip
    T_max
    T_min
    beta_schedule
    gamma_max
    gamma_min
    gamma_schedule

Adam tunables:
    adam_betas
    adam_weight_decay

RMSprop tunables:
    rmsprop_alpha
    rmsprop_momentum
    rmsprop_weight_decay

Example:
    hp = get_k2000_local_hparams(
        n_batch=10000,
        n_rep=1,
        n_steps=1000,
        optimizer="rmsprop",
        lr=3e-2,
        rmsprop_alpha=0.56,
        rmsprop_momentum=0.63,
        rmsprop_weight_decay=0.013,
        grad_type="explicit",
        T_max=1.16,
        T_min=6e-5,
        beta_schedule="linear_temperature",
        gamma_max=1.0,
        gamma_min=1e-3,
        gamma_schedule="exponential",
    )
"""
    )


def get_k2000_local_hparams(
    *,
    n_batch=10000,
    n_rep=1,
    n_steps=1000,

    optimizer="adam",
    lr=None,

    adam_betas=(0.9, 0.999),
    adam_weight_decay=0.0,

    rmsprop_alpha=0.56,
    rmsprop_momentum=0.63,
    rmsprop_weight_decay=0.013,

    grad_type="auto",
    grad_clip=GRAD_CLIP_DEFAULT,

    T_max=1.16,
    T_min=6e-5,
    beta_schedule="linear_temperature",

    gamma_max=1.0,
    gamma_min=1e-3,
    gamma_schedule="exponential",

    show_help=False,
):
    if show_help:
        print_k2000_hparam_help()

    optimizer = optimizer.lower()
    grad_type = grad_type.lower()
    beta_schedule = beta_schedule.lower()
    gamma_schedule = gamma_schedule.lower()

    if optimizer not in ["adam", "rmsprop"]:
        print("ERROR: optimizer must be either 'adam' or 'rmsprop'.")
        print_k2000_hparam_help()
        raise ValueError("Invalid optimizer.")

    if lr is None:
        if optimizer == "adam":
            lr = 1e-2
        elif optimizer == "rmsprop":
            lr = 3e-2

    if grad_type not in ["auto", "explicit"]:
        print("ERROR: grad_type must be either 'auto' or 'explicit'.")
        print_k2000_hparam_help()
        raise ValueError("Invalid grad_type.")

    if beta_schedule not in ["linear_temperature", "linear_beta", "exponential_beta"]:
        print("ERROR: beta_schedule must be one of:")
        print("    'linear_temperature', 'linear_beta', 'exponential_beta'")
        print_k2000_hparam_help()
        raise ValueError("Invalid beta_schedule.")

    if gamma_schedule not in ["linear", "exponential", "constant"]:
        print("ERROR: gamma_schedule must be one of:")
        print("    'linear', 'exponential', 'constant'")
        print_k2000_hparam_help()
        raise ValueError("Invalid gamma_schedule.")

    if n_batch <= 0:
        print("ERROR: n_batch must be positive.")
        print_k2000_hparam_help()
        raise ValueError("Invalid n_batch.")

    if n_rep <= 0:
        print("ERROR: n_rep must be positive.")
        print_k2000_hparam_help()
        raise ValueError("Invalid n_rep.")

    if n_steps <= 0:
        print("ERROR: n_steps must be positive.")
        print_k2000_hparam_help()
        raise ValueError("Invalid n_steps.")

    if lr <= 0:
        print("ERROR: lr must be positive.")
        print_k2000_hparam_help()
        raise ValueError("Invalid lr.")

    if T_max <= 0 or T_min <= 0:
        print("ERROR: T_max and T_min must both be positive.")
        print_k2000_hparam_help()
        raise ValueError("Invalid temperature range.")

    if gamma_max < 0 or gamma_min < 0:
        print("ERROR: gamma_max and gamma_min must be non-negative.")
        print_k2000_hparam_help()
        raise ValueError("Invalid gamma range.")

    return {
        "graph_name": f"K2000_{optimizer}",

        "n_batch": int(n_batch),
        "n_rep": int(n_rep),
        "n_steps": int(n_steps),

        "optimizer": optimizer,
        "lr": float(lr),

        "adam_betas": adam_betas,
        "adam_weight_decay": float(adam_weight_decay),

        "rmsprop_alpha": float(rmsprop_alpha),
        "rmsprop_momentum": float(rmsprop_momentum),
        "rmsprop_weight_decay": float(rmsprop_weight_decay),

        "grad_type": grad_type,
        "gradient_mode": grad_type,
        "grad_clip": grad_clip,

        "T_max": float(T_max),
        "T_min": float(T_min),
        "beta_schedule": beta_schedule,

        "gamma_max": float(gamma_max),
        "gamma_min": float(gamma_min),
        "gamma_schedule": gamma_schedule,
    }

In [18]:
# ============================================================
# REQUIRED CORE QEFEM V2 FUNCTIONS
# Paste this BEFORE the full K2000 + G1-G54 benchmark cell
# ============================================================

import math
import time
import torch
import pandas as pd

# ----------------------------
# Core helpers
# ----------------------------

def logit_from_value(x, device=DEVICE, dtype=DTYPE):
    x = torch.tensor(float(x), device=device, dtype=dtype)
    return torch.logit(x)


def bloch_from_fields(fields, theta):
    R = torch.sigmoid(fields)
    rx = R * torch.sin(theta)
    rz = R * torch.cos(theta)
    return R, rx, rz


def expected_cut(W, rz):
    single_input = False

    if rz.dim() == 1:
        rz = rz.unsqueeze(0)
        single_input = True

    rz = rz.to(device=W.device, dtype=W.dtype)

    quad = torch.einsum("ri,ij,rj->r", rz, W, rz)
    cut = 0.25 * (W.sum() - quad)

    if single_input:
        return cut.squeeze(0)

    return cut


def discrete_cut(W, spins):
    single_input = False

    if spins.dim() == 1:
        spins = spins.unsqueeze(0)
        single_input = True

    spins = spins.to(device=W.device, dtype=W.dtype)

    quad = torch.einsum("ri,ij,rj->r", spins, W, spins)
    cut = 0.25 * (W.sum() - quad)

    if single_input:
        return cut.squeeze(0)

    return cut


def quantum_entropy(R, eps=EPS):
    single_input = False

    if R.dim() == 1:
        R = R.unsqueeze(0)
        single_input = True

    R = R.clamp(eps, 1.0 - eps)

    lam_plus = 0.5 * (1.0 + R)
    lam_minus = 0.5 * (1.0 - R)

    S_site = -(
        lam_plus * torch.log(lam_plus)
        + lam_minus * torch.log(lam_minus)
    )

    S = S_site.sum(dim=1)

    if single_input:
        return S.squeeze(0)

    return S


def transverse_driver_energy(rx, gamma):
    single_input = False

    if rx.dim() == 1:
        rx = rx.unsqueeze(0)
        single_input = True

    gamma = torch.as_tensor(gamma, device=rx.device, dtype=rx.dtype)
    driver = -gamma * rx.sum(dim=1)

    if single_input:
        return driver.squeeze(0)

    return driver


def qefem_free_energy(W, fields, theta, beta, gamma, eps=EPS):
    beta = torch.as_tensor(beta, device=W.device, dtype=W.dtype)
    gamma = torch.as_tensor(gamma, device=W.device, dtype=W.dtype)

    R, rx, rz = bloch_from_fields(fields, theta)

    ecut = expected_cut(W, rz)
    entropy = quantum_entropy(R, eps=eps)
    driver = transverse_driver_energy(rx, gamma)

    internal_energy = -ecut
    F = internal_energy + driver - entropy / beta

    aux = {
        "R": R,
        "rx": rx,
        "rz": rz,
        "expected_cut": ecut,
        "entropy": entropy,
        "driver_energy": driver,
        "internal_energy": internal_energy,
        "free_energy": F,
        "beta": beta,
        "gamma": gamma,
    }

    return F, aux


# ----------------------------
# Schedules
# ----------------------------

def make_beta_schedule(local_hparams, device=DEVICE, dtype=DTYPE):
    n_steps = int(local_hparams["n_steps"])

    T_max = float(local_hparams["T_max"])
    T_min = float(local_hparams["T_min"])

    beta_schedule = local_hparams.get("beta_schedule", "linear_temperature").lower()

    if beta_schedule == "linear_temperature":
        T = torch.linspace(T_max, T_min, n_steps, device=device, dtype=dtype)
        beta = 1.0 / T

    elif beta_schedule == "linear_beta":
        beta_min = 1.0 / T_max
        beta_max = 1.0 / T_min
        beta = torch.linspace(beta_min, beta_max, n_steps, device=device, dtype=dtype)

    elif beta_schedule == "exponential_beta":
        beta_min = 1.0 / T_max
        beta_max = 1.0 / T_min
        beta = torch.exp(
            torch.linspace(
                math.log(beta_min),
                math.log(beta_max),
                n_steps,
                device=device,
                dtype=dtype,
            )
        )

    else:
        raise ValueError("beta_schedule must be linear_temperature, linear_beta, or exponential_beta.")

    return beta


def make_gamma_schedule(local_hparams, device=DEVICE, dtype=DTYPE):
    n_steps = int(local_hparams["n_steps"])

    gamma_max = float(local_hparams["gamma_max"])
    gamma_min = float(local_hparams["gamma_min"])

    gamma_schedule = local_hparams.get("gamma_schedule", "linear").lower()

    if gamma_schedule == "linear":
        gamma = torch.linspace(gamma_max, gamma_min, n_steps, device=device, dtype=dtype)

    elif gamma_schedule == "exponential":
        if gamma_max <= 0 or gamma_min <= 0:
            raise ValueError("exponential gamma requires gamma_max > 0 and gamma_min > 0.")

        gamma = torch.exp(
            torch.linspace(
                math.log(gamma_max),
                math.log(gamma_min),
                n_steps,
                device=device,
                dtype=dtype,
            )
        )

    elif gamma_schedule == "constant":
        gamma = torch.full((n_steps,), gamma_max, device=device, dtype=dtype)

    else:
        raise ValueError("gamma_schedule must be linear, exponential, or constant.")

    return gamma


# ----------------------------
# Optimizer
# ----------------------------

def make_optimizer(params, local_hparams):
    optimizer_name = local_hparams["optimizer"].lower()
    lr = float(local_hparams["lr"])

    if optimizer_name == "adam":
        return torch.optim.Adam(
            params,
            lr=lr,
            betas=local_hparams.get("adam_betas", (0.9, 0.999)),
            weight_decay=float(local_hparams.get("adam_weight_decay", 0.0)),
        )

    if optimizer_name == "rmsprop":
        return torch.optim.RMSprop(
            params,
            lr=lr,
            alpha=float(local_hparams.get("rmsprop_alpha", 0.99)),
            momentum=float(local_hparams.get("rmsprop_momentum", 0.0)),
            weight_decay=float(local_hparams.get("rmsprop_weight_decay", 0.0)),
        )

    raise ValueError("optimizer must be adam or rmsprop.")


def grad_global_norm(params):
    total = torch.zeros((), device=params[0].device)

    for p in params:
        if p.grad is not None:
            total = total + p.grad.detach().pow(2).sum()

    return total.sqrt()


# ----------------------------
# SINGLE INSTANCE RUNNER
# ----------------------------

def run_qefem_single_instance(
    W,
    best_known_cut,
    local_hparams,
    seed=0,
    print_every=50,
    device=DEVICE,
    dtype=DTYPE,
    eps=EPS,
):
    graph_name = local_hparams["graph_name"]
    optimizer_name = local_hparams["optimizer"].lower()
    grad_type = local_hparams.get("grad_type", local_hparams.get("gradient_mode", "auto")).lower()

    if grad_type != "auto":
        raise NotImplementedError("Only grad_type='auto' is implemented in this cell.")

    W = W.to(device=device, dtype=dtype)

    n = W.shape[0]
    n_batch = int(local_hparams["n_batch"])
    n_rep = int(local_hparams["n_rep"])
    n_steps = int(local_hparams["n_steps"])

    beta_values = make_beta_schedule(local_hparams, device=device, dtype=dtype)
    gamma_values = make_gamma_schedule(local_hparams, device=device, dtype=dtype)

    best_cut_so_far = -float("inf")
    best_rep_so_far = None
    best_outer_rep = None
    best_step = None
    best_spins = None

    history = []

    print(f"Starting {graph_name} | historical best = {float(best_known_cut):.0f}")

    t0 = time.time()

    for outer_rep in range(n_rep):
        rep_t0 = time.time()

        torch.manual_seed(seed + outer_rep)

        field_center = logit_from_value(BLOCH_LENGTH_INIT, device=device, dtype=dtype)

        fields = field_center + FIELD_NOISE * torch.randn(
            n_batch, n, device=device, dtype=dtype
        )

        theta = THETA_INIT + THETA_NOISE * torch.randn(
            n_batch, n, device=device, dtype=dtype
        )

        fields.requires_grad_(True)
        theta.requires_grad_(True)

        optimizer = make_optimizer([fields, theta], local_hparams)

        for step in range(n_steps):
            beta = beta_values[step]
            gamma = gamma_values[step]

            optimizer.zero_grad(set_to_none=True)

            F_val, aux = qefem_free_energy(
                W=W,
                fields=fields,
                theta=theta,
                beta=beta,
                gamma=gamma,
                eps=eps,
            )

            loss = F_val.sum()
            loss.backward()

            grad_norm_before = grad_global_norm([fields, theta]).item()

            grad_clip = local_hparams.get("grad_clip", None)
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    [fields, theta],
                    max_norm=float(grad_clip),
                )

            grad_norm_after = grad_global_norm([fields, theta]).item()

            optimizer.step()

            with torch.no_grad():
                R, rx, rz = bloch_from_fields(fields, theta)

                spins = torch.where(
                    rz >= 0,
                    torch.ones_like(rz),
                    -torch.ones_like(rz),
                )

                ecut = expected_cut(W, rz)
                hcut = discrete_cut(W, spins)
                entropy = quantum_entropy(R, eps=eps)

                hard_max = hcut.max()
                hard_argmax = int(torch.argmax(hcut).item())

                if hard_max.item() > best_cut_so_far:
                    best_cut_so_far = float(hard_max.item())
                    best_rep_so_far = hard_argmax
                    best_outer_rep = outer_rep
                    best_step = step
                    best_spins = spins[hard_argmax].detach().cpu().clone()

                rep_elapsed = time.time() - rep_t0
                total_elapsed = time.time() - t0
                global_step = outer_rep * n_steps + step

                gap_now = float(best_known_cut - best_cut_so_far)
                gap_percent_now = 100.0 * gap_now / float(best_known_cut)
                acc_now = 100.0 * best_cut_so_far / float(best_known_cut)

                history.append({
                    "graph_name": graph_name,
                    "optimizer": optimizer_name,
                    "grad_type": grad_type,

                    "outer_rep": outer_rep,
                    "step": step,
                    "global_step": global_step,

                    "beta": float(beta.item()),
                    "gamma": float(gamma.item()),

                    "free_energy_mean": float(F_val.mean().detach().item()),
                    "free_energy_min": float(F_val.min().detach().item()),
                    "free_energy_max": float(F_val.max().detach().item()),

                    "expected_cut_mean": float(ecut.mean().item()),
                    "expected_cut_max": float(ecut.max().item()),
                    "expected_cut_min": float(ecut.min().item()),

                    "hard_cut_mean": float(hcut.mean().item()),
                    "hard_cut_max": float(hcut.max().item()),
                    "hard_cut_min": float(hcut.min().item()),

                    "best_cut_so_far": float(best_cut_so_far),
                    "best_known_cut": float(best_known_cut),
                    "gap_so_far": gap_now,
                    "gap_percent_so_far": gap_percent_now,
                    "accuracy_percent_so_far": acc_now,

                    "best_replica_in_batch": hard_argmax,
                    "best_replica_so_far": best_rep_so_far,
                    "best_outer_rep_so_far": best_outer_rep,
                    "best_step_so_far": best_step,

                    "entropy_mean": float(entropy.mean().item()),
                    "entropy_min": float(entropy.min().item()),
                    "entropy_max": float(entropy.max().item()),

                    "R_mean": float(R.mean().item()),
                    "R_min": float(R.min().item()),
                    "R_max": float(R.max().item()),

                    "rx_abs_mean": float(rx.abs().mean().item()),
                    "rz_abs_mean": float(rz.abs().mean().item()),

                    "grad_norm_before_clip": grad_norm_before,
                    "grad_norm_after_clip": grad_norm_after,

                    "rep_elapsed_sec": float(rep_elapsed),
                    "elapsed_sec": float(total_elapsed),
                })

                if (
                    step == 0
                    or (step + 1) % print_every == 0
                    or step == n_steps - 1
                    or best_cut_so_far >= best_known_cut
                ):
                    print(
                        f"{graph_name} | "
                        f"rep {outer_rep + 1}/{n_rep} | "
                        f"gap {gap_now:.1f} | "
                        f"acc {acc_now:.4f}% | "
                        f"step {step + 1}/{n_steps} | "
                        f"beta {beta.item():.4g} | "
                        f"gamma {gamma.item():.4g} | "
                        f"ecut {ecut.max().item():.1f} | "
                        f"hcut {hcut.max().item():.1f} | "
                        f"best_rep {best_rep_so_far} | "
                        f"S {entropy.mean().item():.4f} | "
                        f"rep_t {rep_elapsed:.1f}s | "
                        f"total_t {total_elapsed:.1f}s"
                    )

                #if best_cut_so_far >= best_known_cut:
                    #break

        del fields, theta, optimizer

        if DEVICE.type == "mps":
            torch.mps.empty_cache()

        #if best_cut_so_far >= best_known_cut:
            #break

    history_df = pd.DataFrame(history)

    elapsed_total = time.time() - t0

    gap = float(best_known_cut - best_cut_so_far)
    gap_percent = 100.0 * gap / float(best_known_cut)
    accuracy = best_cut_so_far / float(best_known_cut)
    accuracy_percent = 100.0 * accuracy

    summary = {
        "graph_name": graph_name,
        "optimizer": optimizer_name,
        "grad_type": grad_type,

        "best_cut": float(best_cut_so_far),
        "best_known_cut": float(best_known_cut),

        "gap": gap,
        "gap_percent": gap_percent,
        "accuracy": accuracy,
        "accuracy_percent": accuracy_percent,
        "success": bool(best_cut_so_far >= best_known_cut),

        "best_outer_rep": best_outer_rep,
        "best_replica": best_rep_so_far,
        "best_step": best_step,

        "n_batch": n_batch,
        "n_rep_requested": n_rep,
        "n_rep_completed": int(history_df["outer_rep"].max() + 1) if len(history_df) else 0,
        "n_steps": n_steps,
        "total_replica_attempts_completed": int((history_df["outer_rep"].max() + 1) * n_batch) if len(history_df) else 0,

        "T_max": float(local_hparams["T_max"]),
        "T_min": float(local_hparams["T_min"]),
        "beta_schedule": local_hparams["beta_schedule"],

        "gamma_max": float(local_hparams["gamma_max"]),
        "gamma_min": float(local_hparams["gamma_min"]),
        "gamma_schedule": local_hparams["gamma_schedule"],

        "lr": float(local_hparams["lr"]),
        "grad_clip": local_hparams.get("grad_clip", None),

        "elapsed_sec": float(elapsed_total),
    }

    return {
        "summary": summary,
        "history": history_df,
        "best_spins": best_spins,
    }

print("run_qefem_single_instance is now defined. Finally. The notebook has been appeased.")

run_qefem_single_instance is now defined. Finally. The notebook has been appeased.


In [17]:
# ============================================================
# FULL QEFEM V2 BENCHMARK RUN: K2000 FIRST, THEN G1-G54
# Saves:
#   full_res/K2000_qefem_v2_anneal_history.csv
#   full_res/K2000_qefem_v2_summary.csv
#   full_res/G{idx}_qefem_v2_anneal_history.csv
#   full_res/G{idx}_qefem_v2_summary.csv
#   full_res/qefem_v2_full_benchmark_table_partial.csv
#   full_res/qefem_v2_full_benchmark_table.csv
# ============================================================

import time
from pathlib import Path
import pandas as pd
import torch

# ----------------------------
# Robust paths because notebooks are little demons
# ----------------------------

CWD = Path.cwd()

OUT_DIR = CWD / "full_res"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_existing_file(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of these paths exist:\n" + "\n".join(str(p) for p in candidates))

def find_existing_dir(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists() and p.is_dir():
            return p
    raise FileNotFoundError("None of these directories exist:\n" + "\n".join(str(p) for p in candidates))

DATA_MAXCUT_DIR = find_existing_dir([
    CWD / "../../data-maxcut",
    CWD / "../data-maxcut",
    CWD / "data-maxcut",
])

GSET_HIST_PATH = find_existing_file([
    CWD / "gset_validation.txt",
    CWD / "../gset_validation.txt",
    CWD / "../../gset_validation.txt",
])

K2000_PATH = find_existing_file([
    DATA_MAXCUT_DIR / "WK2000_1.rud",
    DATA_MAXCUT_DIR / "k2000",
    CWD / "WK2000_1.rud",
    CWD / "../WK2000_1.rud",
])

print("CWD:", CWD)
print("OUT_DIR:", OUT_DIR)
print("DATA_MAXCUT_DIR:", DATA_MAXCUT_DIR)
print("GSET_HIST_PATH:", GSET_HIST_PATH)
print("K2000_PATH:", K2000_PATH)
print("DEVICE:", DEVICE)

# ----------------------------
# Robust loaders
# ----------------------------

def load_gset_dense_v2(g_idx, data_dir=DATA_MAXCUT_DIR, device=DEVICE, dtype=DTYPE):
    g_idx = int(g_idx)
    if not (1 <= g_idx <= 54):
        raise ValueError("g_idx must be between 1 and 54.")

    path = Path(data_dir) / f"G{g_idx}"

    with open(path, "r") as f:
        n, m = map(int, f.readline().split())
        W_cpu = torch.zeros((n, n), dtype=dtype, device="cpu")

        for _ in range(m):
            i, j, w = f.readline().split()
            i = int(i) - 1
            j = int(j) - 1
            w = float(w)

            W_cpu[i, j] = w
            W_cpu[j, i] = w

    return W_cpu.to(device=device)

def load_k2000_dense_v2(path=K2000_PATH, device=DEVICE, dtype=DTYPE):
    path = Path(path)

    with open(path, "r") as f:
        n, m = map(int, f.readline().split())
        W_cpu = torch.zeros((n, n), dtype=dtype, device="cpu")

        for _ in range(m):
            i, j, w = f.readline().split()
            i = int(i) - 1
            j = int(j) - 1
            w = float(w)

            W_cpu[i, j] = w
            W_cpu[j, i] = w

    return W_cpu.to(device=device)

# ----------------------------
# History / best-known table
# ----------------------------

def build_gset_history_df_v2(hist_path=GSET_HIST_PATH):
    rows = []

    with open(hist_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            g_idx, best_cut = line.split()[:2]
            g_idx = int(g_idx)

            if 1 <= g_idx <= 54:
                rows.append({
                    "graph_name": f"G{g_idx}",
                    "best_known_cut": float(best_cut),
                })

    df = pd.DataFrame(rows).sort_values(
        by="graph_name",
        key=lambda x: x.str.replace("G", "", regex=False).astype(int)
    ).reset_index(drop=True)

    df.to_csv(OUT_DIR / "gset_history_best_G1-G54.csv", index=False)
    return df

gset_history_df = build_gset_history_df_v2()

# ----------------------------
# Preferred V2 hyperparams
# ----------------------------

def get_v2_gset_benchmark_hparams(g_idx, n_batch=10000):
    return get_gset_local_hparams(
        int(g_idx),

        n_batch=int(n_batch),
        n_rep=1,
        n_steps=2000,

        optimizer="adam",
        lr=0.01,

        adam_betas=(0.9, 0.999),
        adam_weight_decay=0.0,

        rmsprop_alpha=0.99,
        rmsprop_momentum=0.0,
        rmsprop_weight_decay=0.0,

        grad_type="auto",
        grad_clip=1.0,

        T_max=0.5,
        T_min=8e-5,
        beta_schedule="linear_temperature",

        gamma_max=1.0,
        gamma_min=1e-3,
        gamma_schedule="linear",
    )

def get_v2_k2000_benchmark_hparams(n_batch=10000):
    return get_k2000_local_hparams(
        n_batch=int(n_batch),
        n_rep=1,
        n_steps=2000,

        optimizer="adam",
        lr=0.01,

        adam_betas=(0.9, 0.999),
        adam_weight_decay=0.0,

        rmsprop_alpha=0.99,
        rmsprop_momentum=0.0,
        rmsprop_weight_decay=0.0,

        grad_type="auto",
        grad_clip=1.0,

        T_max=0.5,
        T_min=8e-5,
        beta_schedule="linear_temperature",

        gamma_max=1.0,
        gamma_min=1e-3,
        gamma_schedule="linear",
    )

# ----------------------------
# Summary normalization
# Similar to V1 comparison metrics
# ----------------------------

def normalize_v2_summary(summary, n_nodes):
    best_known = float(summary["best_known_cut"])
    best_cut = float(summary["best_cut"])

    gap_abs = best_known - best_cut
    gap_percent = 100.0 * gap_abs / best_known
    accuracy_percent = 100.0 * best_cut / best_known

    return {
        "graph_name": summary["graph_name"],
        "n_nodes": int(n_nodes),

        "best_known_cut": best_known,
        "qefem_v2_best_cut": best_cut,

        "v2_gap": gap_abs,
        "v2_gap_percent": gap_percent,
        "v2_accuracy_percent": accuracy_percent,
        "success": bool(best_cut >= best_known),

        "optimizer": summary.get("optimizer", None),
        "grad_type": summary.get("grad_type", None),

        "n_batch": summary.get("n_batch", None),
        "n_rep_requested": summary.get("n_rep_requested", None),
        "n_rep_completed": summary.get("n_rep_completed", None),
        "n_steps": summary.get("n_steps", None),

        "lr": summary.get("lr", None),
        "grad_clip": summary.get("grad_clip", None),

        "T_max": summary.get("T_max", None),
        "T_min": summary.get("T_min", None),
        "beta_schedule": summary.get("beta_schedule", None),

        "gamma_max": summary.get("gamma_max", None),
        "gamma_min": summary.get("gamma_min", None),
        "gamma_schedule": summary.get("gamma_schedule", None),

        "best_outer_rep": summary.get("best_outer_rep", None),
        "best_replica": summary.get("best_replica", None),
        "best_step": summary.get("best_step", None),

        "elapsed_sec": summary.get("elapsed_sec", None),
    }

# ----------------------------
# Run one instance and save
# ----------------------------

def run_and_save_one_v2(graph_name, W, best_known_cut, local_hparams, print_every=50):
    print("=" * 110)
    print(f"STARTING {graph_name} | historical best = {best_known_cut:.0f}")
    print("=" * 110)

    result = run_qefem_single_instance(
        W=W,
        best_known_cut=float(best_known_cut),
        local_hparams=local_hparams,
        seed=0,
        print_every=print_every,
    )

    history_df = result["history"].copy()
    summary = result["summary"].copy()

    history_path = OUT_DIR / f"{graph_name}_qefem_v2_anneal_history.csv"
    summary_path = OUT_DIR / f"{graph_name}_qefem_v2_summary.csv"

    history_df.to_csv(history_path, index=False)
    pd.DataFrame([summary]).to_csv(summary_path, index=False)

    row = normalize_v2_summary(summary, n_nodes=W.shape[0])
    row["history_path"] = str(history_path)
    row["summary_path"] = str(summary_path)

    print(f"FINISHED {graph_name}")
    print(
        f"best = {row['qefem_v2_best_cut']:.0f} | "
        f"gap = {row['v2_gap']:.0f} | "
        f"gap% = {row['v2_gap_percent']:.4f}% | "
        f"acc = {row['v2_accuracy_percent']:.4f}% | "
        f"time = {row['elapsed_sec']:.2f}s"
    )
    print()

    return row

# ----------------------------
# FULL RUN: K2000 first, then G1-G54
# ----------------------------

all_rows = []

FULL_RUN_T0 = time.time()

# ---- K2000 first ----

W_K2000 = load_k2000_dense_v2()
k2000_hparams = get_v2_k2000_benchmark_hparams(n_batch=10000)

k2000_row = run_and_save_one_v2(
    graph_name="K2000",
    W=W_K2000,
    best_known_cut=K2000_HIST_BEST,
    local_hparams=k2000_hparams,
    print_every=50,
)

all_rows.append(k2000_row)

benchmark_df = pd.DataFrame(all_rows)
benchmark_df.to_csv(OUT_DIR / "qefem_v2_full_benchmark_table_partial.csv", index=False)

del W_K2000
if DEVICE.type == "mps":
    torch.mps.empty_cache()

# ---- G1-G54 next ----

for g_idx in range(1, 55):
    graph_name = f"G{g_idx}"

    best_known_cut = float(
        gset_history_df.loc[
            gset_history_df["graph_name"] == graph_name,
            "best_known_cut"
        ].iloc[0]
    )

    W = load_gset_dense_v2(g_idx)
    local_hparams = get_v2_gset_benchmark_hparams(g_idx, n_batch=10000)

    row = run_and_save_one_v2(
        graph_name=graph_name,
        W=W,
        best_known_cut=best_known_cut,
        local_hparams=local_hparams,
        print_every=50,
    )

    all_rows.append(row)

    benchmark_df = pd.DataFrame(all_rows)
    benchmark_df.to_csv(OUT_DIR / "qefem_v2_full_benchmark_table_partial.csv", index=False)

    del W
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

# ---- final benchmark table ----

benchmark_df = pd.DataFrame(all_rows)

ordered_cols = [
    "graph_name",
    "n_nodes",
    "best_known_cut",
    "qefem_v2_best_cut",
    "v2_gap",
    "v2_gap_percent",
    "v2_accuracy_percent",
    "success",
    "optimizer",
    "grad_type",
    "n_batch",
    "n_rep_requested",
    "n_rep_completed",
    "n_steps",
    "lr",
    "grad_clip",
    "T_max",
    "T_min",
    "beta_schedule",
    "gamma_max",
    "gamma_min",
    "gamma_schedule",
    "best_outer_rep",
    "best_replica",
    "best_step",
    "elapsed_sec",
    "history_path",
    "summary_path",
]

benchmark_df = benchmark_df[[c for c in ordered_cols if c in benchmark_df.columns]]

benchmark_df.to_csv(OUT_DIR / "qefem_v2_full_benchmark_table.csv", index=False)

print("=" * 110)
print("FULL QEFEM V2 BENCHMARK COMPLETE")
print(f"Total elapsed = {time.time() - FULL_RUN_T0:.2f}s")
print(f"Saved final table to: {OUT_DIR / 'qefem_v2_full_benchmark_table.csv'}")
print(f"Saved partial table to: {OUT_DIR / 'qefem_v2_full_benchmark_table_partial.csv'}")
print("=" * 110)

benchmark_df

CWD: /Users/ronit/Desktop/QeFEM/main/maxcut_results/v2
OUT_DIR: /Users/ronit/Desktop/QeFEM/main/maxcut_results/v2/full_res
DATA_MAXCUT_DIR: /Users/ronit/Desktop/QeFEM/main/maxcut_results/v2/../../data-maxcut
GSET_HIST_PATH: /Users/ronit/Desktop/QeFEM/main/maxcut_results/v2/gset_validation.txt
K2000_PATH: /Users/ronit/Desktop/QeFEM/main/maxcut_results/v2/../../data-maxcut/k2000
DEVICE: mps
STARTING K2000 | historical best = 33337
Starting K2000_adam | historical best = 33337
K2000_adam | rep 1/1 | gap 37234.0 | acc -11.6897% | step 1/2000 | beta 2 | gamma 1 | ecut -520.0 | hcut -3897.0 | best_rep 4499 | S 1386.1924 | rep_t 0.7s | total_t 0.7s
K2000_adam | rep 1/1 | gap 30448.0 | acc 8.6660% | step 50/2000 | beta 2.05 | gamma 0.9755 | ecut -520.0 | hcut -1735.0 | best_rep 8021 | S 1386.0259 | rep_t 37.0s | total_t 37.0s
K2000_adam | rep 1/1 | gap 5603.0 | acc 83.1928% | step 100/2000 | beta 2.104 | gamma 0.9505 | ecut -520.0 | hcut 27734.0 | best_rep 2724 | S 1385.5801 | rep_t 83.0s | to

KeyboardInterrupt: 

In [19]:
# ============================================================
# CONTINUE / RUN ONLY G1-G54
# Does NOT rerun K2000
# Skips any Gset instance whose history CSV already exists
# ============================================================

from pathlib import Path
import pandas as pd
import torch
import time

OUT_DIR = Path.cwd() / "full_res"
OUT_DIR.mkdir(parents=True, exist_ok=True)

gset_history_df = build_gset_history_df_v2()

gset_rows = []

partial_path = OUT_DIR / "qefem_v2_G1-G54_benchmark_table_partial.csv"
final_path = OUT_DIR / "qefem_v2_G1-G54_benchmark_table.csv"

if partial_path.exists():
    gset_rows = pd.read_csv(partial_path).to_dict("records")

already_done = set(row["graph_name"] for row in gset_rows)

for g_idx in range(1, 55):
    graph_name = f"G{g_idx}"

    history_path = OUT_DIR / f"{graph_name}_qefem_v2_anneal_history.csv"
    summary_path = OUT_DIR / f"{graph_name}_qefem_v2_summary.csv"

    if history_path.exists() and summary_path.exists():
        print(f"SKIPPING {graph_name} | already saved")
        if graph_name not in already_done:
            summary = pd.read_csv(summary_path).iloc[0].to_dict()
            row = normalize_v2_summary(summary, n_nodes=int(pd.read_csv(history_path, nrows=1).shape[0]) if False else 0)
            row["graph_name"] = graph_name
            row["history_path"] = str(history_path)
            row["summary_path"] = str(summary_path)
            gset_rows.append(row)
        continue

    best_known_cut = float(
        gset_history_df.loc[
            gset_history_df["graph_name"] == graph_name,
            "best_known_cut"
        ].iloc[0]
    )

    W = load_gset_dense_v2(g_idx)

    local_hparams = get_v2_gset_benchmark_hparams(
        g_idx=g_idx,
        n_batch=10000,
    )

    row = run_and_save_one_v2(
        graph_name=graph_name,
        W=W,
        best_known_cut=best_known_cut,
        local_hparams=local_hparams,
        print_every=50,
    )

    gset_rows.append(row)

    pd.DataFrame(gset_rows).to_csv(partial_path, index=False)

    del W

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

gset_benchmark_df = pd.DataFrame(gset_rows)
gset_benchmark_df.to_csv(final_path, index=False)

print("=" * 100)
print("G1-G54 RUN COMPLETE")
print(f"Saved: {final_path}")
print("=" * 100)

gset_benchmark_df

SKIPPING G1 | already saved
STARTING G2 | historical best = 11620
Starting G2 | historical best = 11620
G2 | rep 1/1 | gap 1830.0 | acc 84.2513% | step 1/2000 | beta 2 | gamma 1 | ecut 9588.0 | hcut 9790.0 | best_rep 288 | S 554.4769 | rep_t 0.8s | total_t 0.8s
G2 | rep 1/1 | gap 1677.0 | acc 85.5680% | step 50/2000 | beta 2.05 | gamma 0.9755 | ecut 9588.0 | hcut 9704.0 | best_rep 4994 | S 554.4104 | rep_t 11.9s | total_t 11.9s
G2 | rep 1/1 | gap 1630.0 | acc 85.9725% | step 100/2000 | beta 2.104 | gamma 0.9505 | ecut 9588.0 | hcut 9882.0 | best_rep 5273 | S 554.2321 | rep_t 23.8s | total_t 23.8s
G2 | rep 1/1 | gap 1630.0 | acc 85.9725% | step 150/2000 | beta 2.161 | gamma 0.9255 | ecut 9588.0 | hcut 9863.0 | best_rep 5273 | S 553.7673 | rep_t 35.4s | total_t 35.4s
G2 | rep 1/1 | gap 1630.0 | acc 85.9725% | step 200/2000 | beta 2.221 | gamma 0.9005 | ecut 9588.0 | hcut 9839.0 | best_rep 5273 | S 552.5869 | rep_t 50.7s | total_t 50.7s
G2 | rep 1/1 | gap 1630.0 | acc 85.9725% | step 250/

KeyboardInterrupt: 